# U04_C2 — Formats and Conversion
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 4:** GenBank Sequence Database
**Type:** Course notebook · **Time:** 35 min · **Difficulty:** ★★☆
**Goes with:** Lectures L7–L9, Practical S4 (EMBOSS seqret & Format Conversion)
**Before this:** U04_C1

## 1. What you will learn
- **Differentiate** format (syntax) from biological content (semantics) [LO2].
- **Convert** sequence records between GenBank, FASTA, and EMBL formats using `Bio.SeqIO` [LO2].
- **Recognize** information loss when converting richly annotated records into simple formats [LO2].
- **Distinguish** primary sequence archives (GenBank/ENA/DDBJ) from curated secondary databases (RefSeq, UniProt) [LO1, LO2].


## 2. Biological question
Bioinformaticians constantly shuttle sequences between different tools. BLAST requires FASTA format; phylogenetic software often requires Clustal or PHYLIP; genome browsers consume GFF3 or GenBank. If you convert a GenBank record into FASTA to run BLAST, what metadata did you discard? Can you recover the lost information by converting it back?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U04_C2"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Format vs. Content
### Theory
Biological sequences can be serialized into many distinct syntax formats:
- **FASTA**: Minimalist format. Header line (`>id description`) followed by raw sequence. Stores *no* annotations, no features, and no coordinates.
- **GenBank**: NCBI flat file. Stores full provenance, taxonomy, citations, and coordinate-mapped feature tables.
- **EMBL**: European Molecular Biology Laboratory flat file. Uses two-letter line codes (`ID`, `AC`, `DE`, `KW`, `OS`, `FT`, `SQ`).
- **GFF3 / GTF**: Tabular format specialized for genome annotations.

Converting from a rich format (GenBank) to a simple format (FASTA) is a **lossy conversion**.


In [ ]:
# Worked Example: Converting GenBank to FASTA and EMBL
import io
from Bio import SeqIO

gb_path = DATA_DIR / "genbank" / "hbb_human.gb"
record = SeqIO.read(gb_path, "genbank")

# Write to in-memory text buffers to inspect formatting
fasta_buffer = io.StringIO()
SeqIO.write(record, fasta_buffer, "fasta")
fasta_text = fasta_buffer.getvalue()

print("=== Converted FASTA Format (First 4 lines) ===")
print("\n".join(fasta_text.splitlines()[:4]))
print("...")

embl_buffer = io.StringIO()
SeqIO.write(record, embl_buffer, "embl")
embl_text = embl_buffer.getvalue()

print("\n=== Converted EMBL Format (First 6 lines) ===")
print("\n".join(embl_text.splitlines()[:6]))


## 4. Concept 2: Demonstrating Information Loss
### Theory
When we write a `SeqRecord` loaded from GenBank to a FASTA file and re-read it:
- The sequence and ID are preserved.
- All entries in `record.features` (genes, exons, CDS, variations) are **discarded**.
- All entries in `record.annotations` (organism, taxonomy, citations) are **discarded**.


In [ ]:
# Worked Example: Quantifying lost information
# Save to outputs directory
out_fasta = OUT_DIR / "hbb_converted.fasta"
SeqIO.write(record, out_fasta, "fasta")

reloaded_fasta = SeqIO.read(out_fasta, "fasta")

print("Original GenBank Record:")
print("  Features count:   ", len(record.features))
print("  Annotations count:", len(record.annotations))

print("\nReloaded FASTA Record:")
print("  Features count:   ", len(reloaded_fasta.features))
print("  Annotations count:", len(reloaded_fasta.annotations))


## 5. Concept 3: Primary vs. Secondary Databases
### Theory
- **Primary Databases (Archives)**: GenBank, ENA, DDBJ. Direct user submissions. Redundant; authors retain ownership; errors are not edited by database staff.
- **Secondary Databases (Curated)**: RefSeq, UniProtKB/Swiss-Prot. Curated by expert biocurators. Non-redundant; explicit reference standards.
- Human *HBB* primary submissions include `V00497.1` (cloned in 1978 by Lawn et al.). Its secondary RefSeq standard is `NM_000518.5`.


In [ ]:
# Worked Example: Checking primary source in RefSeq record
comment = record.annotations.get("comment", "")
print("RefSeq provenance comment:")
print(" ", comment[:150], "...")


## 6. Compare with EMBOSS seqret
In Practical S4, we ran `seqret -sequence hbb_human.gb -outseq hbb.fasta` from the command line.
Biopython's `SeqIO.write(record, handle, "fasta")` performs the exact same operation programmatically.


## 7. Summary
- `Bio.SeqIO.write(records, file_handle, format)` exports sequence records to standard biological formats.
- Converting GenBank to FASTA strips annotations and features irreversibly.
- Primary databases store original submitted historical data; secondary databases curate standardized benchmarks.


## 8. Next Steps
- Complete practice exercises on INS and TP53 in `U04_P_GenBank_and_Formats.ipynb`.
- Take the Unit 4 quiz in `U04_Q_GenBank_Database.ipynb`.
